# 02 Estimate Screening Cost

Estimate the screening cost for the configured LLM and Jev/System 1 approaches over candidate `(PMID, pathogen, antibiotic, year)` rows.

This notebook uses the upstream candidate, abstract-cache, and prediction parquets to estimate only rows that still need predictions. If the candidate parquet is a smaller pilot subset, `cost_estimation.project_to_configured_pathogens` scales observed candidate volume to `collection.num_pathogens`. It does not call either model. Set `cost_estimation.exclude_already_asked` to `False` to estimate the full uncached run.

LLM pricing comes from `cost_estimation.pricing_by_model`; Jev pricing comes from `cost_estimation.system1_pricing_by_model`. The Jev token count is approximate because the provider does not expose its tokenizer here. Cached-input discounts are route-dependent and not assumed by default; configure a cached-input fraction and rate only when you have evidence for them. Check current Jev rates at [OpenRouter](https://openrouter.ai/typesafe/jev-1.13/api).

## Inputs

- Candidate rows from `candidate_amr_pair_year_counts.parquet`.
- Abstracts and no-abstract cache records from `pmid_abstracts.parquet` (coverage must be complete; run 04a if it is not).
- Existing LLM decisions and the configured current System 1 run, used to exclude completed predictions.
- Model pricing, tokenization, and projection settings from `outputs/pipeline_config.yaml`.

## Outputs

- In-notebook cost and call estimates for both LLM and System 1, including their configured prices and cached-input assumptions. This notebook does not write predictions or call either model.


In [1]:
import math
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.screening import (LLM_CLASSIFICATION_MAX_TOKENS, UNIFIED_AMR_INTEREST_SYSTEM_PROMPT, SYSTEM1_AMR_SUPPORT_QUESTION, build_system1_state, build_user_prompt, prepare_yearly_candidate_pmids)
from amr.pipeline_config import cfg, cfg_path, load_pipeline_config


In [2]:
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)

OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.output_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.parquet_dir", PROJECT_ROOT)
DATA_DIR = PARQUET_OUTPUT_DIR

YEARLY_CANDIDATE_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_candidates", PROJECT_ROOT)
ABSTRACT_CACHE_PATH = cfg_path(PIPELINE_CONFIG, "paths.abstract_cache", PROJECT_ROOT)
UNIFIED_DECISIONS_PATH = cfg_path(PIPELINE_CONFIG, "paths.unified_decisions", PROJECT_ROOT)
SYSTEM1_DECISIONS_PATH = cfg_path(PIPELINE_CONFIG, "paths.system1_decisions", PROJECT_ROOT)
ARCHIVED_DECISIONS_GLOB = OUTPUT_DIR / "archive" / "*" / "yearly_amr_unified_decisions.parquet"

MODEL_NAME = cfg(PIPELINE_CONFIG, "llm.model")
SYSTEM1_MODEL = cfg(PIPELINE_CONFIG, "system1.model", "typesafe/jev-1.13")
SYSTEM1_RUN_ID = str(cfg(PIPELINE_CONFIG, "system1.run_id", "jev-default"))

MODEL_PRICING_BY_MODEL = cfg(PIPELINE_CONFIG, "cost_estimation.pricing_by_model", {})
if not isinstance(MODEL_PRICING_BY_MODEL, dict):
    raise TypeError("cost_estimation.pricing_by_model must be a mapping from model names to token prices.")
MODEL_PRICING_KEY = MODEL_NAME if MODEL_NAME in MODEL_PRICING_BY_MODEL else next(
    (key for key in sorted(MODEL_PRICING_BY_MODEL, key=len, reverse=True) if str(MODEL_NAME).startswith(f"{key}-")),
    None,
)
if MODEL_PRICING_KEY is None:
    raise KeyError(
        f"No pricing configured for llm.model={MODEL_NAME!r}. "
        f"Add it under cost_estimation.pricing_by_model; available models: {sorted(MODEL_PRICING_BY_MODEL)}"
    )
MODEL_PRICING = MODEL_PRICING_BY_MODEL[MODEL_PRICING_KEY]
INPUT_USD_PER_1M_TOKENS = float(MODEL_PRICING["input_usd_per_1m_tokens"])
OUTPUT_USD_PER_1M_TOKENS = float(MODEL_PRICING["output_usd_per_1m_tokens"])
CACHED_INPUT_USD_PER_1M_TOKENS = float(MODEL_PRICING["cached_input_usd_per_1m_tokens"])

# Batch multiplier: keep 1.0 for ordinary synchronous API calls; adjust for Batch API pricing if used.
BATCH_API_MULTIPLIER = cfg(PIPELINE_CONFIG, "cost_estimation.batch_api_multiplier")

# Token estimation controls.
SAMPLE_SIZE = cfg(PIPELINE_CONFIG, "cost_estimation.sample_size")
RANDOM_SEED = cfg(PIPELINE_CONFIG, "cost_estimation.random_seed")
ASSUMED_OUTPUT_TOKENS_PER_CALL = cfg(PIPELINE_CONFIG, "cost_estimation.assumed_output_tokens_per_call")
USE_CACHED_INPUT_PRICE = cfg(PIPELINE_CONFIG, "cost_estimation.use_cached_input_price")
EXCLUDE_ALREADY_ASKED = bool(cfg(PIPELINE_CONFIG, "cost_estimation.exclude_already_asked", True))
PROJECT_TO_CONFIGURED_PATHOGENS = bool(cfg(PIPELINE_CONFIG, "cost_estimation.project_to_configured_pathogens", True))
TARGET_NUM_PATHOGENS = cfg(PIPELINE_CONFIG, "collection.num_pathogens")

SYSTEM1_PRICING_BY_MODEL = cfg(PIPELINE_CONFIG, "cost_estimation.system1_pricing_by_model", {})
if not isinstance(SYSTEM1_PRICING_BY_MODEL, dict):
    raise TypeError("cost_estimation.system1_pricing_by_model must map model names to token prices.")
SYSTEM1_PRICING_KEY = SYSTEM1_MODEL if SYSTEM1_MODEL in SYSTEM1_PRICING_BY_MODEL else next(
    (key for key in sorted(SYSTEM1_PRICING_BY_MODEL, key=len, reverse=True) if str(SYSTEM1_MODEL).startswith(f"{key}-")),
    None,
)
if SYSTEM1_PRICING_KEY is None:
    raise KeyError(
        f"No pricing configured for system1.model={SYSTEM1_MODEL!r}. "
        f"Add it under cost_estimation.system1_pricing_by_model; available models: {sorted(SYSTEM1_PRICING_BY_MODEL)}"
    )
SYSTEM1_PRICING = SYSTEM1_PRICING_BY_MODEL[SYSTEM1_PRICING_KEY]
SYSTEM1_INPUT_USD_PER_1M_TOKENS = float(SYSTEM1_PRICING["input_usd_per_1m_tokens"])
SYSTEM1_OUTPUT_USD_PER_1M_TOKENS = float(SYSTEM1_PRICING.get("output_usd_per_1m_tokens", 0.0))
SYSTEM1_CACHED_INPUT_USD_PER_1M_TOKENS = SYSTEM1_PRICING.get("cached_input_usd_per_1m_tokens")
SYSTEM1_CACHED_INPUT_FRACTION = float(cfg(PIPELINE_CONFIG, "cost_estimation.system1_cached_input_fraction", 0.0))
if not 0.0 <= SYSTEM1_CACHED_INPUT_FRACTION <= 1.0:
    raise ValueError("cost_estimation.system1_cached_input_fraction must be between 0 and 1.")
if SYSTEM1_CACHED_INPUT_FRACTION and SYSTEM1_CACHED_INPUT_USD_PER_1M_TOKENS is None:
    raise ValueError(
        "Set cached_input_usd_per_1m_tokens for this Jev model before using a non-zero "
        "system1_cached_input_fraction."
    )
if SYSTEM1_CACHED_INPUT_USD_PER_1M_TOKENS is not None:
    SYSTEM1_CACHED_INPUT_USD_PER_1M_TOKENS = float(SYSTEM1_CACHED_INPUT_USD_PER_1M_TOKENS)
SYSTEM1_PROBABILITY_THRESHOLD = float(cfg(PIPELINE_CONFIG, "screening.system1_probability_threshold", 0.5))


## Load Candidate Rows and Abstracts

The LLM and Jev are estimated independently at candidate-row granularity: one row per `(PMID, pathogen, antibiotic, year)`. Rows without abstracts do not call either model. Previously completed predictions are excluded separately for each source and the configured Jev run ID.


In [3]:
if not YEARLY_CANDIDATE_PATH.exists():
    raise FileNotFoundError(f"Missing {YEARLY_CANDIDATE_PATH}. Run notebook 04 collection first.")
if not ABSTRACT_CACHE_PATH.exists():
    raise FileNotFoundError(f"Missing {ABSTRACT_CACHE_PATH}. Run notebook 04 abstract prefetch/filtering first.")

yearly_candidate_df = pd.read_parquet(YEARLY_CANDIDATE_PATH)
candidate_df = prepare_yearly_candidate_pmids(yearly_candidate_df)
abstract_df = pd.read_parquet(ABSTRACT_CACHE_PATH)
required_abstract_cols = {"pmid", "title", "abstract"}
missing_abstract_cols = required_abstract_cols - set(abstract_df.columns)
if missing_abstract_cols:
    raise KeyError(f"{ABSTRACT_CACHE_PATH} missing required columns: {sorted(missing_abstract_cols)}")

abstract_df = abstract_df[["pmid", "title", "abstract"]].copy()
abstract_df["pmid"] = abstract_df["pmid"].astype(str).str.strip()
candidate_pmids = set(candidate_df["pmid"].dropna().astype(str).str.strip())
cached_pmids = set(abstract_df["pmid"].dropna())
missing_cache_pmids = sorted(candidate_pmids - cached_pmids)
if missing_cache_pmids:
    examples = ", ".join(missing_cache_pmids[:20])
    raise RuntimeError(
        f"Abstract cache coverage is incomplete: {len(missing_cache_pmids):,} of "
        f"{len(candidate_pmids):,} candidate PMIDs are missing. Run notebook "
        f"04a_prepare_candidate_abstracts.ipynb, then rerun this estimate. "
        f"Examples: {examples}"
    )
abstract_df["pmid"] = abstract_df["pmid"].astype(str)
abstract_df["title"] = abstract_df["title"].fillna("").astype(str)
abstract_df["abstract"] = abstract_df["abstract"].fillna("").astype(str)

review_df = candidate_df.merge(abstract_df, on="pmid", how="left")
review_df["title"] = review_df["title"].fillna("").astype(str)
review_df["abstract"] = review_df["abstract"].fillna("").astype(str)
candidate_key_cols = ["pmid", "pathogen", "antibiotic", "year"]
for key_col in ["pmid", "pathogen", "antibiotic"]:
    review_df[key_col] = review_df[key_col].astype(str).str.strip()
review_df["year"] = pd.to_numeric(review_df["year"], errors="coerce").astype("Int64")
already_asked_df = pd.DataFrame(columns=candidate_key_cols)
if EXCLUDE_ALREADY_ASKED and UNIFIED_DECISIONS_PATH.exists():
    existing_decisions_df = pd.read_parquet(UNIFIED_DECISIONS_PATH)
    missing_decision_keys = set(candidate_key_cols) - set(existing_decisions_df.columns)
    if missing_decision_keys:
        raise KeyError(f"{UNIFIED_DECISIONS_PATH} is missing decision key columns: {sorted(missing_decision_keys)}")
    already_asked_df = existing_decisions_df[candidate_key_cols].drop_duplicates().copy()
    for key_col in ["pmid", "pathogen", "antibiotic"]:
        already_asked_df[key_col] = already_asked_df[key_col].astype(str).str.strip()
    already_asked_df["year"] = pd.to_numeric(already_asked_df["year"], errors="coerce").astype("Int64")
review_df = review_df.merge(
    already_asked_df.assign(_already_asked=True),
    on=candidate_key_cols,
    how="left",
).assign(_already_asked=lambda df: df["_already_asked"].fillna(False))
already_asked_mask = review_df["_already_asked"].astype(bool)
has_abstract_mask = review_df["abstract"].str.strip().ne("")
eligible_mask = (~already_asked_mask) if EXCLUDE_ALREADY_ASKED else pd.Series(True, index=review_df.index)
llm_call_df = review_df.loc[has_abstract_mask & eligible_mask].copy()
no_abstract_df = review_df.loc[~has_abstract_mask & eligible_mask].copy()

system1_already_asked_df = pd.DataFrame(columns=candidate_key_cols)
if EXCLUDE_ALREADY_ASKED and SYSTEM1_DECISIONS_PATH.exists():
    existing_system1_df = pd.read_parquet(SYSTEM1_DECISIONS_PATH)
    required_system1_cols = set(candidate_key_cols) | {"system1_run_id", "system1_status"}
    missing_system1_cols = required_system1_cols - set(existing_system1_df.columns)
    if missing_system1_cols:
        raise KeyError(f"{SYSTEM1_DECISIONS_PATH} is missing columns: {sorted(missing_system1_cols)}")
    existing_system1_df = existing_system1_df.loc[
        (existing_system1_df["system1_run_id"].astype(str) == SYSTEM1_RUN_ID)
        & (existing_system1_df["system1_status"].astype(str).isin({"ok", "no_abstract"}))
    ]
    system1_already_asked_df = existing_system1_df[candidate_key_cols].drop_duplicates().copy()
    for key_col in ["pmid", "pathogen", "antibiotic"]:
        system1_already_asked_df[key_col] = system1_already_asked_df[key_col].astype(str).str.strip()
    system1_already_asked_df["year"] = pd.to_numeric(
        system1_already_asked_df["year"], errors="coerce"
    ).astype("Int64")
review_df = review_df.merge(
    system1_already_asked_df.assign(_system1_already_asked=True),
    on=candidate_key_cols,
    how="left",
).assign(_system1_already_asked=lambda df: df["_system1_already_asked"].fillna(False))
system1_already_asked_mask = review_df["_system1_already_asked"].astype(bool)
system1_eligible_mask = (~system1_already_asked_mask) if EXCLUDE_ALREADY_ASKED else pd.Series(True, index=review_df.index)
system1_call_df = review_df.loc[has_abstract_mask & system1_eligible_mask].copy()
no_abstract_system1_df = review_df.loc[~has_abstract_mask & system1_eligible_mask].copy()
actual_candidate_pathogens = int(candidate_df["pathogen"].nunique())
if TARGET_NUM_PATHOGENS is not None and int(TARGET_NUM_PATHOGENS) < 1:
    raise ValueError("collection.num_pathogens must be positive or None.")
projection_scale = 1.0
if PROJECT_TO_CONFIGURED_PATHOGENS and TARGET_NUM_PATHOGENS is not None and actual_candidate_pathogens:
    projection_scale = max(1.0, float(TARGET_NUM_PATHOGENS) / actual_candidate_pathogens)
projected_candidate_rows = math.ceil(len(candidate_df) * projection_scale)
projected_abstract_rows = math.ceil(int(has_abstract_mask.sum()) * projection_scale)
already_asked_cost_mask = already_asked_mask & has_abstract_mask
already_asked_cost_rows = int(already_asked_cost_mask.sum()) if EXCLUDE_ALREADY_ASKED else 0
estimated_llm_calls = (
    max(projected_abstract_rows - already_asked_cost_rows, 0)
    if projection_scale > 1.0
    else len(llm_call_df)
)
system1_already_asked_cost_rows = (
    int((system1_already_asked_mask & has_abstract_mask).sum()) if EXCLUDE_ALREADY_ASKED else 0
)
estimated_system1_calls = (
    max(projected_abstract_rows - system1_already_asked_cost_rows, 0)
    if projection_scale > 1.0
    else len(system1_call_df)
)
estimated_no_abstract_rows = max(projected_candidate_rows - projected_abstract_rows, 0)
token_sample_source_df = review_df.loc[has_abstract_mask].copy() if projection_scale > 1.0 else llm_call_df
system1_token_sample_source_df = review_df.loc[has_abstract_mask].copy() if projection_scale > 1.0 else system1_call_df

summary_df = pd.DataFrame(
    [
        {"metric": "candidate_rows_observed", "value": len(candidate_df)},
        {"metric": "candidate_pathogens_observed", "value": actual_candidate_pathogens},
        {"metric": "target_pathogens", "value": TARGET_NUM_PATHOGENS if TARGET_NUM_PATHOGENS is not None else "all"},
        {"metric": "projection_scale", "value": projection_scale},
        {"metric": "candidate_rows_projected", "value": projected_candidate_rows},
        {"metric": "llm_rows_already_asked_excluded", "value": int(already_asked_mask.sum()) if EXCLUDE_ALREADY_ASKED else 0},
        {"metric": "system1_rows_already_scored_excluded", "value": int(system1_already_asked_mask.sum()) if EXCLUDE_ALREADY_ASKED else 0},
        {"metric": "rows_with_abstract_observed", "value": int(has_abstract_mask.sum())},
        {"metric": "estimated_llm_calls", "value": estimated_llm_calls},
        {"metric": "estimated_system1_calls", "value": estimated_system1_calls},
        {"metric": "estimated_rows_without_abstract_or_cache", "value": estimated_no_abstract_rows},
        {"metric": "unique_pmids_in_candidates", "value": candidate_df["pmid"].nunique()},
        {"metric": "unique_pmids_with_abstract_observed", "value": review_df.loc[has_abstract_mask, "pmid"].nunique()},
    ]
)
summary_df


,metric,value
0,candidate_rows_observed,186724
1,candidate_pathogens_observed,402
2,target_pathogens,all
3,projection_scale,1.0
4,candidate_rows_projected,186724
5,llm_rows_already_asked_excluded,186724
6,system1_rows_already_scored_excluded,0
7,rows_with_abstract_observed,186276
8,estimated_llm_calls,0
9,estimated_system1_calls,186276


## Token Counter

`tiktoken` is used when available. If it is not installed, the notebook falls back to a conservative character-based estimate. The same available tokenizer is used as a proxy for Jev; its exact tokenizer and request framing are not exposed here.


In [4]:
try:
    import tiktoken

    try:
        encoding = tiktoken.encoding_for_model(MODEL_NAME)
    except KeyError:
        encoding = tiktoken.get_encoding("o200k_base")

    TOKENIZER = f"tiktoken:{encoding.name}"

    def count_tokens(text: str) -> int:
        return len(encoding.encode(text or ""))

except Exception as exc:
    TOKENIZER = f"chars_per_4_fallback:{exc.__class__.__name__}"

    def count_tokens(text: str) -> int:
        return max(1, int(round(len(text or "") / 4)))

TOKENIZER


'tiktoken:o200k_base'

In [5]:
def prompt_token_count(row: pd.Series) -> int:
    user_prompt = build_user_prompt(
        pmid=str(row["pmid"]),
        pathogen=str(row["pathogen"]),
        antibiotic=str(row["antibiotic"]),
        title=str(row.get("title", "")),
        abstract=str(row.get("abstract", "")),
    )
    # Chat message framing adds a small overhead. This keeps the estimate conservative without depending on API internals.
    message_overhead_tokens = 12
    return count_tokens(UNIFIED_AMR_INTEREST_SYSTEM_PROMPT) + count_tokens(user_prompt) + message_overhead_tokens

def system1_input_token_count(row: pd.Series) -> int:
    state = build_system1_state(
        pmid=str(row["pmid"]),
        pathogen=str(row["pathogen"]),
        antibiotic=str(row["antibiotic"]),
        title=str(row.get("title", "")),
        abstract=str(row.get("abstract", "")),
    )
    # Approximate request framing for the state, one noul question, and response schema.
    return count_tokens(state) + count_tokens(SYSTEM1_AMR_SUPPORT_QUESTION) + 20

sample_n = min(SAMPLE_SIZE, len(token_sample_source_df))
sample_df = token_sample_source_df.sample(n=sample_n, random_state=RANDOM_SEED).copy()
sample_df["input_tokens"] = 0.0
if sample_n:
    sample_df["input_tokens"] = sample_df.apply(prompt_token_count, axis=1)

system1_sample_n = min(SAMPLE_SIZE, len(system1_token_sample_source_df))
system1_sample_df = system1_token_sample_source_df.sample(
    n=system1_sample_n, random_state=RANDOM_SEED
).copy()
system1_sample_df["input_tokens"] = 0.0
if system1_sample_n:
    system1_sample_df["input_tokens"] = system1_sample_df.apply(system1_input_token_count, axis=1)

input_token_stats = sample_df["input_tokens"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_frame("input_tokens")
system1_input_token_stats = system1_sample_df["input_tokens"].describe(
    percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
).to_frame("system1_input_tokens")
print("LLM input-token sample:")
display(input_token_stats)
print("System 1 state + question-token sample (approximate):")
system1_input_token_stats


LLM input-token sample:


,input_tokens
count,0.0
mean,NaN
std,NaN
min,NaN
5%,NaN
25%,NaN
50%,NaN
75%,NaN
95%,NaN
max,NaN


System 1 state + question-token sample (approximate):


,system1_input_tokens
count,1000.000000
mean,591.965000
std,134.206219
min,248.000000
5%,405.950000
25%,501.000000
50%,578.000000
75%,659.250000
95%,836.050000
max,1235.000000


## Estimate Output Tokens

LLM output tokens are inferred from existing decision JSON or `ASSUMED_OUTPUT_TOKENS_PER_CALL`. Jev returns a typed answer with negligible output-token charges under the configured OpenRouter pricing; this estimate applies its configured output rate to zero output tokens.


In [6]:
import json

def clean_bool(value: object, default: bool = False) -> bool:
    if pd.isna(value):
        return default
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes"}

def clean_list(value: object) -> list[str]:
    if isinstance(value, list):
        return [str(item) for item in value if str(item).strip()]
    if hasattr(value, "tolist"):
        return clean_list(value.tolist())
    return []

def clean_text(value: object, default: str = "") -> str:
    if pd.isna(value):
        return default
    return str(value)

def compact_json_for_decision(row: pd.Series) -> str:
    payload = {
        "decision": clean_text(row.get("unified_decision", row.get("decision", "uncertain")), "uncertain"),
        "primary_category": clean_text(row.get("amr_primary_category", "NONE"), "NONE"),
        "rationale": clean_text(row.get("unified_rationale", row.get("rationale", ""))),
        "evidence_phrases": clean_list(row.get("amr_evidence_phrases", [])),
        "target_pair_relation_supported": clean_bool(row.get("amr_target_pair_relation_supported", False)),
        "is_amr_relevant": clean_bool(row.get("amr_is_relevant", False)),
    }
    return json.dumps(payload, ensure_ascii=False, separators=(",", ":"))

candidate_decision_paths = []
if UNIFIED_DECISIONS_PATH.exists():
    candidate_decision_paths.append(UNIFIED_DECISIONS_PATH)
candidate_decision_paths.extend(sorted(OUTPUT_DIR.glob("archive/*/yearly_amr_unified_decisions.parquet")))

output_token_source = "fallback_assumption"
estimated_output_tokens_per_call = ASSUMED_OUTPUT_TOKENS_PER_CALL
output_token_stats = pd.DataFrame()

for decision_path in candidate_decision_paths:
    try:
        decisions_df = pd.read_parquet(decision_path)
    except Exception:
        continue
    if decisions_df.empty:
        continue
    sample_decisions = decisions_df.sample(n=min(SAMPLE_SIZE, len(decisions_df)), random_state=RANDOM_SEED).copy()
    token_counts = sample_decisions.apply(lambda row: count_tokens(compact_json_for_decision(row)), axis=1)
    if len(token_counts):
        estimated_output_tokens_per_call = float(token_counts.median())
        output_token_source = str(decision_path)
        output_token_stats = token_counts.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_frame("output_tokens")
        break

print(f"output_token_source: {output_token_source}")
print(f"estimated_output_tokens_per_call: {estimated_output_tokens_per_call:.1f}")
output_token_stats


output_token_source: /Users/f.costa/Code/Antimicrobial-Resistance/outputs/parquets/yearly_amr_unified_decisions.parquet
estimated_output_tokens_per_call: 38.0


,output_tokens
count,1000.000000
mean,53.031000
std,40.655097
min,35.000000
5%,35.000000
25%,36.000000
50%,38.000000
75%,38.250000
95%,164.000000
max,329.000000


## Extrapolate Total Cost

The sampled mean input tokens are multiplied by the number of rows that will actually call the LLM. Output tokens use the inferred or assumed per-call estimate above.


In [7]:
input_price = CACHED_INPUT_USD_PER_1M_TOKENS if USE_CACHED_INPUT_PRICE else INPUT_USD_PER_1M_TOKENS
calls = estimated_llm_calls
mean_input_tokens = float(sample_df["input_tokens"].mean()) if calls else 0.0
median_input_tokens = float(sample_df["input_tokens"].median()) if calls else 0.0
p95_input_tokens = float(sample_df["input_tokens"].quantile(0.95)) if calls else 0.0

total_input_tokens_mean = calls * mean_input_tokens
total_input_tokens_median = calls * median_input_tokens
total_input_tokens_p95 = calls * p95_input_tokens
total_output_tokens = calls * estimated_output_tokens_per_call

def usd(input_tokens: float, output_tokens: float) -> float:
    return BATCH_API_MULTIPLIER * (
        (input_tokens / 1_000_000) * input_price
        + (output_tokens / 1_000_000) * OUTPUT_USD_PER_1M_TOKENS
    )

cost_summary_df = pd.DataFrame(
    [
        {
            "scenario": "mean_input_tokens",
            "llm_calls": calls,
            "input_tokens_per_call": mean_input_tokens,
            "output_tokens_per_call": estimated_output_tokens_per_call,
            "total_input_tokens": total_input_tokens_mean,
            "total_output_tokens": total_output_tokens,
            "estimated_cost_usd": usd(total_input_tokens_mean, total_output_tokens),
        },
        {
            "scenario": "median_input_tokens",
            "llm_calls": calls,
            "input_tokens_per_call": median_input_tokens,
            "output_tokens_per_call": estimated_output_tokens_per_call,
            "total_input_tokens": total_input_tokens_median,
            "total_output_tokens": total_output_tokens,
            "estimated_cost_usd": usd(total_input_tokens_median, total_output_tokens),
        },
        {
            "scenario": "p95_input_tokens_upper_bound",
            "llm_calls": calls,
            "input_tokens_per_call": p95_input_tokens,
            "output_tokens_per_call": estimated_output_tokens_per_call,
            "total_input_tokens": total_input_tokens_p95,
            "total_output_tokens": total_output_tokens,
            "estimated_cost_usd": usd(total_input_tokens_p95, total_output_tokens),
        },
    ]
)

cost_summary_df["estimated_cost_usd"] = cost_summary_df["estimated_cost_usd"].round(4)
cost_summary_df


system1_calls = estimated_system1_calls
system1_mean_input_tokens = float(system1_sample_df["input_tokens"].mean()) if system1_calls else 0.0
system1_median_input_tokens = float(system1_sample_df["input_tokens"].median()) if system1_calls else 0.0
system1_p95_input_tokens = float(system1_sample_df["input_tokens"].quantile(0.95)) if system1_calls else 0.0

def system1_usd(input_tokens: float, output_tokens: float = 0.0) -> float:
    cached_tokens = input_tokens * SYSTEM1_CACHED_INPUT_FRACTION
    uncached_tokens = input_tokens - cached_tokens
    cached_input_price = (
        SYSTEM1_CACHED_INPUT_USD_PER_1M_TOKENS
        if SYSTEM1_CACHED_INPUT_USD_PER_1M_TOKENS is not None
        else SYSTEM1_INPUT_USD_PER_1M_TOKENS
    )
    return BATCH_API_MULTIPLIER * (
        (uncached_tokens / 1_000_000) * SYSTEM1_INPUT_USD_PER_1M_TOKENS
        + (cached_tokens / 1_000_000) * cached_input_price
        + (output_tokens / 1_000_000) * SYSTEM1_OUTPUT_USD_PER_1M_TOKENS
    )

system1_cost_summary_df = pd.DataFrame(
    [
        {
            "scenario": scenario,
            "system1_calls": system1_calls,
            "input_tokens_per_call": tokens_per_call,
            "output_tokens_per_call": float("nan"),
            "total_input_tokens": system1_calls * tokens_per_call,
            "total_output_tokens": float("nan"),
            "estimated_cost_usd": system1_usd(system1_calls * tokens_per_call),
        }
        for scenario, tokens_per_call in [
            ("mean_input_tokens", system1_mean_input_tokens),
            ("median_input_tokens", system1_median_input_tokens),
            ("p95_input_tokens_upper_bound", system1_p95_input_tokens),
        ]
    ]
)

llm_comparison_df = cost_summary_df.assign(
    prediction_source="llm",
    model=MODEL_NAME,
    calls=cost_summary_df["llm_calls"],
).drop(columns=["llm_calls"])
system1_comparison_df = system1_cost_summary_df.assign(
    prediction_source="system1",
    model=SYSTEM1_MODEL,
    calls=system1_cost_summary_df["system1_calls"],
).drop(columns=["system1_calls"])
cost_comparison_df = pd.concat([llm_comparison_df, system1_comparison_df], ignore_index=True)
cost_comparison_df["estimated_cost_usd"] = cost_comparison_df["estimated_cost_usd"].round(4)
cost_comparison_df[[
    "prediction_source", "model", "scenario", "calls", "input_tokens_per_call",
    "total_input_tokens", "total_output_tokens", "estimated_cost_usd",
]]


,prediction_source,model,scenario,calls,input_tokens_per_call,total_input_tokens,total_output_tokens,estimated_cost_usd
0,llm,gpt-5-nano,mean_input_tokens,0,0.000,0.000000e+00,0.0,0.0000
1,llm,gpt-5-nano,median_input_tokens,0,0.000,0.000000e+00,0.0,0.0000
2,llm,gpt-5-nano,p95_input_tokens_upper_bound,0,0.000,0.000000e+00,0.0,0.0000
3,system1,typesafe/jev-1.13,mean_input_tokens,186276,591.965,1.102689e+08,NaN,4.6313
4,system1,typesafe/jev-1.13,median_input_tokens,186276,578.000,1.076675e+08,NaN,4.5220
5,system1,typesafe/jev-1.13,p95_input_tokens_upper_bound,186276,836.050,1.557360e+08,NaN,6.5409


In [8]:
print(f"LLM model: {MODEL_NAME} (pricing key: {MODEL_PRICING_KEY})")
print(f"System 1 model: {SYSTEM1_MODEL} (pricing key: {SYSTEM1_PRICING_KEY}, run ID: {SYSTEM1_RUN_ID})")
print(f"Tokenizer/estimator: {TOKENIZER} (Jev token counts are approximate)")
print(f"LLM input price used: ${input_price:.6f} per 1M tokens")
print(f"LLM output price used: ${OUTPUT_USD_PER_1M_TOKENS:.6f} per 1M tokens")
print(f"System 1 input price used: ${SYSTEM1_INPUT_USD_PER_1M_TOKENS:.6f} per 1M tokens")
print(f"System 1 output price used: ${SYSTEM1_OUTPUT_USD_PER_1M_TOKENS:.6f} per 1M tokens")
print(f"System 1 cached-input fraction: {SYSTEM1_CACHED_INPUT_FRACTION:.0%}")
pricing_comparison_df = pd.DataFrame(
    [
        {
            "prediction_source": "llm",
            "model": MODEL_NAME,
            "input_usd_per_1m_tokens": INPUT_USD_PER_1M_TOKENS,
            "cached_input_usd_per_1m_tokens": CACHED_INPUT_USD_PER_1M_TOKENS,
            "cached_input_fraction_assumed": 1.0 if USE_CACHED_INPUT_PRICE else 0.0,
            "input_price_used_usd_per_1m_tokens": input_price,
            "output_usd_per_1m_tokens": OUTPUT_USD_PER_1M_TOKENS,
        },
        {
            "prediction_source": "system1",
            "model": SYSTEM1_MODEL,
            "input_usd_per_1m_tokens": SYSTEM1_INPUT_USD_PER_1M_TOKENS,
            "cached_input_usd_per_1m_tokens": SYSTEM1_CACHED_INPUT_USD_PER_1M_TOKENS,
            "cached_input_fraction_assumed": SYSTEM1_CACHED_INPUT_FRACTION,
            "input_price_used_usd_per_1m_tokens": (
                SYSTEM1_INPUT_USD_PER_1M_TOKENS * (1 - SYSTEM1_CACHED_INPUT_FRACTION)
                + (SYSTEM1_CACHED_INPUT_USD_PER_1M_TOKENS or SYSTEM1_INPUT_USD_PER_1M_TOKENS)
                * SYSTEM1_CACHED_INPUT_FRACTION
            ),
            "output_usd_per_1m_tokens": SYSTEM1_OUTPUT_USD_PER_1M_TOKENS,
        },
    ]
)
print("Configured token prices and cache assumptions ($ per 1M tokens):")
display(pricing_comparison_df)
print(f"Batch multiplier: {BATCH_API_MULTIPLIER}")
print(f"Observed candidate pathogens: {actual_candidate_pathogens:,}")
print(f"Target pathogens: {TARGET_NUM_PATHOGENS if TARGET_NUM_PATHOGENS is not None else 'all'}")
print(f"Projection scale: {projection_scale:.2f}x")
print(f"Projected candidate rows: {projected_candidate_rows:,}")
print(f"LLM rows already asked and excluded: {int(already_asked_mask.sum()) if EXCLUDE_ALREADY_ASKED else 0:,}")
print(f"System 1 rows already scored for this run and excluded: {int(system1_already_asked_mask.sum()) if EXCLUDE_ALREADY_ASKED else 0:,}")
print(f"Rows that call the LLM: {estimated_llm_calls:,}")
print(f"Rows that call Jev: {estimated_system1_calls:,}")
print(f"Projected rows skipped because abstract is missing: {estimated_no_abstract_rows:,}")
print("Estimated cost comparison:")
cost_comparison_df[["prediction_source", "scenario", "calls", "estimated_cost_usd"]]


LLM model: gpt-5-nano (pricing key: gpt-5-nano)
System 1 model: typesafe/jev-1.13 (pricing key: typesafe/jev-1.13, run ID: jev-1.13-amr-support-v1)
Tokenizer/estimator: tiktoken:o200k_base (Jev token counts are approximate)
LLM input price used: $0.050000 per 1M tokens
LLM output price used: $0.400000 per 1M tokens
System 1 input price used: $0.042000 per 1M tokens
System 1 output price used: $0.000000 per 1M tokens
System 1 cached-input fraction: 0%
Configured token prices and cache assumptions ($ per 1M tokens):


,prediction_source,model,input_usd_per_1m_tokens,cached_input_usd_per_1m_tokens,cached_input_fraction_assumed,input_price_used_usd_per_1m_tokens,output_usd_per_1m_tokens
0,llm,gpt-5-nano,0.050,0.005,0.0,0.050,0.4
1,system1,typesafe/jev-1.13,0.042,NaN,0.0,0.042,0.0


Batch multiplier: 1.0
Observed candidate pathogens: 402
Target pathogens: all
Projection scale: 1.00x
Projected candidate rows: 186,724
LLM rows already asked and excluded: 186,724
System 1 rows already scored for this run and excluded: 0
Rows that call the LLM: 0
Rows that call Jev: 186,276
Projected rows skipped because abstract is missing: 448
Estimated cost comparison:


,prediction_source,scenario,calls,estimated_cost_usd
0,llm,mean_input_tokens,0,0.0000
1,llm,median_input_tokens,0,0.0000
2,llm,p95_input_tokens_upper_bound,0,0.0000
3,system1,mean_input_tokens,186276,4.6313
4,system1,median_input_tokens,186276,4.5220
5,system1,p95_input_tokens_upper_bound,186276,6.5409


## Optional: Inspect Largest Estimated Prompts

Use this to sanity-check whether unusually long abstracts dominate the estimate.


In [9]:
inspect_cols = ["pmid", "pathogen", "antibiotic", "year", "input_tokens", "title"]
sample_df.sort_values("input_tokens", ascending=False)[inspect_cols].head(20)


,pmid,pathogen,antibiotic,year,input_tokens,title
